In [1]:
# 자연어 입력 >> SQL 문 >> 실행 >> 결과값 >> 자연어 형태로 사용자에게 전달

import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql",
    filename="Chinook_Sqlite.sql",
)

('Chinook_Sqlite.sql', <http.client.HTTPMessage at 0x7b1b55122a10>)

In [2]:
!pip install langchain_community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 19.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.6/974.6 kB 42.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.8/321.8 kB 27.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.0/145.0 kB 10.7 MB/s eta 0:00:00


데이터베이스 구축

In [5]:
import sqlite3

# 데이터베이스 연결
# 데이터베이스가 없으면 >> 생성
# /content/Chinook_Sqlite.sql
conn = sqlite3.connect('drwill.db')

with open('Chinook_Sqlite.sql', 'r') as file:
    script = file.read()

# 스크립트 실행
conn.executescript(script)

conn.close()

LLM 환경 설정

In [6]:
!pip install openai
!pip install langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.5/325.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.4 MB/s eta 0:00:00


In [7]:
import os

In [8]:
from langchain.chat_models import ChatOpenAI

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

chat_model = ChatOpenAI()


DB 연결

In [9]:
# Langchain 환경에서 sql db와 상호작용할 수 있게 함

from langchain.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///drwill.db")

In [11]:
print(db.get_table_info())


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Empl

In [12]:
# 체인으로 구성하기 위해, 인자가 하나인 함수 구성
# 만약 인자가 없으면 에러 발생

def get_schema(_):
  print("###")
  print(_)
  print("in get_schema")
  print("###")

  return db.get_table_info()

In [13]:
get_schema(_)

###

CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)

/*
3 rows from Album table:
AlbumId	Title	ArtistId
1	For Those About To Rock We Salute You	1
2	Balls to the Wall	2
3	Restless and Wild	2
*/


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	AC/DC
2	Accept
3	Aerosmith
*/


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "

'\nCREATE TABLE "Album" (\n\t"AlbumId" INTEGER NOT NULL, \n\t"Title" NVARCHAR(160) NOT NULL, \n\t"ArtistId" INTEGER NOT NULL, \n\tPRIMARY KEY ("AlbumId"), \n\tFOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")\n)\n\n/*\n3 rows from Album table:\nAlbumId\tTitle\tArtistId\n1\tFor Those About To Rock We Salute You\t1\n2\tBalls to the Wall\t2\n3\tRestless and Wild\t2\n*/\n\n\nCREATE TABLE "Artist" (\n\t"ArtistId" INTEGER NOT NULL, \n\t"Name" NVARCHAR(120), \n\tPRIMARY KEY ("ArtistId")\n)\n\n/*\n3 rows from Artist table:\nArtistId\tName\n1\tAC/DC\n2\tAccept\n3\tAerosmith\n*/\n\n\nCREATE TABLE "Customer" (\n\t"CustomerId" INTEGER NOT NULL, \n\t"FirstName" NVARCHAR(40) NOT NULL, \n\t"LastName" NVARCHAR(20) NOT NULL, \n\t"Company" NVARCHAR(80), \n\t"Address" NVARCHAR(70), \n\t"City" NVARCHAR(40), \n\t"State" NVARCHAR(40), \n\t"Country" NVARCHAR(40), \n\t"PostalCode" NVARCHAR(10), \n\t"Phone" NVARCHAR(24), \n\t"Fax" NVARCHAR(24), \n\t"Email" NVARCHAR(60) NOT NULL, \n\t"SupportRepId" INTEG

자연어를 sql 문으로 변환

In [14]:
from langchain.prompts import ChatPromptTemplate

template = """Based on the table schema below,
write a SQL query that would answer the user's question:

{schema}

Question: {question}
SQL Query:"""

# SQL Query: 가 왜 필요할까요?
# 답변 유도문 + 응답형식 지정

chat_prompt_template = ChatPromptTemplate.from_template(template)

In [17]:
# 모델의 출력 >> 문자열로 parsing
from langchain.schema.output_parser import StrOutputParser

from langchain.schema.runnable import RunnablePassthrough

sql_gen_chain = (
    RunnablePassthrough.assign(schema=get_schema)
    # 기존 dict 에다가 schema key 와 value 추가
    | chat_prompt_template
    | chat_model
    | StrOutputParser()

)

In [18]:
sql_gen_chain.invoke({"question" : "How many employees are there?"})

###
{'question': 'How many employees are there?'}
in get_schema
###


'SELECT COUNT(*) AS totalEmployees\nFROM Employee;'

In [19]:
sql_gen_chain.invoke({"question" : "총 고객 수는?"})

###
{'question': '총 고객 수는?'}
in get_schema
###


'SELECT COUNT(*) AS TotalCustomers\nFROM Customer;'

In [20]:
sql_gen_chain.invoke({"question" : "가장 매출이 높은 음반은?"})

###
{'question': '가장 매출이 높은 음반은?'}
in get_schema
###


'SELECT a.Title AS AlbumTitle, SUM(il.UnitPrice * il.Quantity) AS TotalRevenue\nFROM Album a\nJOIN Track t ON a.AlbumId = t.AlbumId\nJOIN InvoiceLine il ON t.TrackId = il.TrackId\nGROUP BY a.AlbumId\nORDER BY TotalRevenue DESC\nLIMIT 1;'

In [21]:
db

In [22]:
#  chain 수정

sql_gen_execute_chain = (
    RunnablePassthrough.assign(schema=get_schema)
    # 기존 dict 에다가 schema key 와 value 추가
    | chat_prompt_template
    | chat_model
    | StrOutputParser()
    | db.run

)

sql_gen_execute_chain = sql_gen_chain | db.run

In [23]:
sql_gen_execute_chain.invoke({"question": "총 고객의 수는?"})
# [(59,)]

###
{'question': '총 고객의 수는?'}
in get_schema
###


'[(59,)]'

In [30]:
db.run('SELECT COUNT(CustomerId) AS TotalCustomers\nFROM Customer;')

'[(59,)]'

SQL 문의 실행결과를 사용자에게 전달하는 체인 구성

In [32]:
template = """Based on the table schema below, question, sql query, and sql response, write a natural language response in Korean:
{schema}

Question: {question}
SQL Query: {query}
SQL Response: {response}"""

chat_prompt_response = ChatPromptTemplate.from_template(template)

In [34]:
from langchain.schema.output_parser import StrOutputParser

'''
sql_response_chain = (
        RunnablePassthrough.assign(schema = get_schema) # 기존 dict에다가 schema 키와 값을 추가한다.
        | prompt_template
        | chat_model
        | StrOutputParser()
    )
'''

full_chain = (
    RunnablePassthrough.assign(query = sql_gen_chain)
    # sql을 생성해주는 chain >> query 라는 key 할당
    | RunnablePassthrough.assign(
        schema = get_schema,
        response = lambda x : db.run(x['query']),
        # db 실행 결과 반환

    )
    | chat_prompt_response
    | chat_model
    | StrOutputParser()
    # 응답 >> 자연어 문자열로 파싱
)

In [35]:
full_chain.invoke({"question": "How many employees are there?"})

###
{'question': 'How many employees are there?'}
in get_schema
###
###
{'question': 'How many employees are there?', 'query': 'SELECT COUNT(EmployeeId) AS TotalEmployees\nFROM Employee;'}
in get_schema
###


'질문: 직원은 몇 명입니까?\nSQL 쿼리: SELECT COUNT(EmployeeId) AS TotalEmployees\nFROM Employee;\nSQL 응답: [(8,)] \n\n답변: 총 8명의 직원이 있습니다.'

In [36]:
ans = full_chain.invoke({"question": "How many employees are there?"})

###
{'question': 'How many employees are there?'}
in get_schema
###
###
{'question': 'How many employees are there?', 'query': 'SELECT COUNT(EmployeeId) AS TotalEmployees\nFROM Employee;'}
in get_schema
###


In [37]:
ans

'질문: 직원은 몇 명입니까?\nSQL 질의: SELECT COUNT(EmployeeId) AS TotalEmployees\nFROM Employee;\nSQL 응답: [(8,)] \n\n답변: 총 8명의 직원이 있습니다.'

In [40]:
def llm(input_text):
   output = full_chain.invoke({"question": input_text})
  #  print(output)

   return output

In [41]:
llm("직원은 몇 명이야?")

###
{'question': '직원은 몇 명이야?'}
in get_schema
###
###
{'question': '직원은 몇 명이야?', 'query': 'SELECT COUNT(EmployeeId) AS NumberOfEmployees\nFROM Employee;'}
in get_schema
###


'직원은 8명입니다.'

In [42]:
def chat_with_user(user_message):
  ai_message = llm(user_message)
  return ai_message

while True:
   user_message = input("user >>")

   if user_message.lower() == "quit":
     break
   ai_message = chat_with_user(user_message)
   print(f'ai >> {ai_message}')

user >>데이터 테이블은 인쇄하는 코드?
###
{'question': '데이터 테이블은 인쇄하는 코드?'}
in get_schema
###
###
{'question': '데이터 테이블은 인쇄하는 코드?', 'query': 'SELECT * FROM Album;'}
in get_schema
###
ai >> 앨범 테이블에서 3개의 행을 출력하였습니다. 총 3개의 앨범이 있으며, AC/DC와 Accept 그리고 Aerosmith가 각각의 앨범을 가지고 있습니다.
user >>quit
